In [1]:
import pandas as pd
import numpy as np

# Creating the dataset
data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                          'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                          'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                          'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                      560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                  3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                      33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                        'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                        'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                        'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)

# --- Task 1: Load and Inspect ---

# 1. Display basic information
print("--- DataFrame Info ---")
df.info()

print("\n--- Missing Values Count ---")
# 2. Count missing values in each column
print(df.isna().sum())

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB

--- Missing Values Count ---
transaction_id      0
date                0
region              3
product_category    2
sales_amount        4
quantity            3
customer_age        4
payment_method      3
dtype: int64


### Task 2: Handle Missing Values

Now, let's clean the dataset by handling missing values as specified:

In [6]:
# 1. Region & Product_category: Fill with mode
df['region'] = df['region'].fillna(df['region'].mode()[0])
df['product_category'] = df['product_category'].fillna(df['product_category'].mode()[0])

# 2. Sales_amount: Fill with median
df['sales_amount'] = df['sales_amount'].fillna(df['sales_amount'].median())

# 3. Quantity: Fill using forward fill (ffill)
df['quantity'] = df['quantity'].ffill()

# 4. Customer_age: Fill with mean (rounded to integer)
df['customer_age'] = df['customer_age'].fillna(round(df['customer_age'].mean()))

# 5. Payment_method: Drop rows where payment_method is missing
df.dropna(subset=['payment_method'], inplace=True)

print("--- Missing Values After Cleaning ---")
# Verify no missing values remain
print(df.isna().sum())

--- Missing Values After Cleaning ---
transaction_id      0
date                0
region              0
product_category    0
sales_amount        0
quantity            0
customer_age        0
payment_method      0
dtype: int64


### Task 3: GroupBy Analysis

In [10]:
# 1. Calculate total sales by region
total_sales_by_region = df.groupby('region')['sales_amount'].sum().reset_index()
print("--- Total Sales by Region ---")
print(total_sales_by_region)

--- Total Sales by Region ---
  region  sales_amount
0   East        3790.0
1  North        6460.0
2  South        1900.0
3   West        2230.0


In [11]:
# 2. Calculate average sales by product_category
average_sales_by_product = df.groupby('product_category')['sales_amount'].mean().reset_index()
print("\n--- Average Sales by Product Category ---")
print(average_sales_by_product)


--- Average Sales by Product Category ---
  product_category  sales_amount
0            Books    508.333333
1         Clothing    680.000000
2      Electronics   1381.250000
3             Home    812.500000


In [12]:
# 3. Group by both region and product_category, calculate total sales and quantity
regional_product_summary = df.groupby(['region', 'product_category']).agg(
    total_sales=('sales_amount', 'sum'),
    total_quantity=('quantity', 'sum')
).reset_index()
print("\n--- Regional Product Summary (Total Sales and Quantity) ---")
display(regional_product_summary)

# 4. Display top 3 region-product combinations by sales
top_3_combinations = regional_product_summary.sort_values(by='total_sales', ascending=False).head(3)
print("\n--- Top 3 Region-Product Combinations by Sales ---")
print(top_3_combinations)


--- Regional Product Summary (Total Sales and Quantity) ---


,region,product_category,total_sales,total_quantity
0,East,Books,800.0,4.0
1,East,Clothing,890.0,3.0
2,East,Electronics,2100.0,1.0
3,North,Clothing,510.0,3.0
4,North,Electronics,2700.0,3.0
5,North,Home,3250.0,12.0
6,South,Clothing,1900.0,9.0
7,West,Books,725.0,1.0
8,West,Clothing,780.0,5.0
9,West,Electronics,725.0,1.0



--- Top 3 Region-Product Combinations by Sales ---
  region product_category  total_sales  total_quantity
5  North             Home       3250.0            12.0
4  North      Electronics       2700.0             3.0
2   East      Electronics       2100.0             1.0


### Task 4: Custom Aggregation

In [15]:
# 1. Create a function sales_range() that returns max - min of sales
def sales_range(series):
    return series.max() - series.min()

# Apply it to find sales range for each region
sales_range_by_region = df.groupby('region')['sales_amount'].agg(sales_range).reset_index()
print("--- Sales Range by Region ---")
print(sales_range_by_region)

--- Sales Range by Region ---
  region  sales_amount
0   East        1720.0
1  North         990.0
2  South         275.0
3   West          55.0


In [17]:
# 2. Use .agg() to calculate multiple metrics by region:
# sales_amount: sum, mean, max
# quantity: sum, min
multiple_metrics_by_region = df.groupby('region').agg(
    total_sales=('sales_amount', 'sum'),
    average_sales=('sales_amount', 'mean'),
    max_sales=('sales_amount', 'max'),
    total_quantity=('quantity', 'sum'),
    min_quantity=('quantity', 'min')
).reset_index()
print("\n--- Multiple Metrics by Region ---")
print(multiple_metrics_by_region)


--- Multiple Metrics by Region ---
  region  total_sales  average_sales  max_sales  total_quantity  min_quantity
0   East       3790.0     947.500000     2100.0             8.0           1.0
1  North       6460.0     922.857143     1500.0            18.0           1.0
2  South       1900.0     633.333333      725.0             9.0           2.0
3   West       2230.0     743.333333      780.0             7.0           1.0


As you can see, all missing values have been successfully handled according to the specified strategies.